# Fashion MNIST Project: Feature Extraction & Classification

This notebook combines the entire pipeline for the Fashion MNIST project, including:
1.  **Data Loading**: Downloading and preparing the dataset.
2.  **Feature Extraction**: Using a pre-trained ResNet18 model to extract features from images.
3.  **Classification (Logistic Regression)**: Training a classifier on the extracted features.
4.  **Clustering (K-Means)**: Unsupervised clustering of the features.

## 1. Setup and Imports

In [ ]:
# Install requirements if not already installed
# !pip install torch torchvision scikit-learn matplotlib seaborn pandas tqdm numpy

import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score, precision_recall_fscore_support, silhouette_score
from sklearn.preprocessing import label_binarize
from sklearn.cluster import KMeans
from scipy.stats import mode

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Utility Functions
Helper functions for plotting and metrics.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix', filename=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
    plt.show()

def plot_roc_curve(y_true, y_prob, n_classes, classes, title='ROC Curve', filename=None):
    # Binarize the output
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f'Class {classes[i]} (area = {roc_auc[i]:.2f})')
        
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
    plt.show()

def plot_accuracy_vs_c(c_values, accuracies, title='Validation Accuracy vs C', filename=None):
    plt.figure(figsize=(8, 6))
    plt.plot(c_values, accuracies, marker='o')
    plt.xscale('log')
    plt.xlabel('C (Inverse Regularization Strength)')
    plt.ylabel('Validation Accuracy')
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
    plt.show()

def print_metrics(y_true, y_pred, classes):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    
    print(f"Accuracy: {accuracy:.4f}")
    
    df = pd.DataFrame({
        'Class': classes,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })
    print("\nClass-wise Metrics:")
    print(df.to_string(index=False))
    return accuracy, df

## 3. Data Loading

In [ ]:
def get_data_loaders(batch_size=32, data_dir='./data'):
    """
    Downloads and prepares the Fashion-MNIST dataset.
    Returns DataLoaders for train, validation, and test sets.
    """
    
    # Define transformations
    # Resize to 224x224 for ResNet
    # Convert to 3 channels (grayscale -> RGB)
    # Normalize with ImageNet mean and std
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Download datasets
    full_train_dataset = datasets.FashionMNIST(root=data_dir, train=True, download=True, transform=transform)
    test_dataset = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)

    # Split training into train (50k) and validation (10k)
    train_size = 50000
    val_size = 10000
    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

## 4. Feature Extraction
We use a pre-trained ResNet18 model (removing the final classification layer) to extract features from the images.

In [ ]:
def extract_features(loader, model, device, set_name="Set", quick=False):
    """
    Extracts features from a data loader using the provided model.
    Returns features and labels as numpy arrays.
    """
    model.eval()
    features_list = []
    labels_list = []

    with torch.inference_mode():
        for i, (inputs, labels) in enumerate(tqdm(loader, desc=f"Extracting features ({set_name})")):
            inputs = inputs.to(device)
            outputs = model(inputs)
            outputs = outputs.view(outputs.size(0), -1)

            features_list.append(outputs.cpu().numpy())
            labels_list.append(labels.numpy())

            # Quick mode: only process first few batches
            if quick and i >= 5:
                break

    features = np.concatenate(features_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return features, labels

# Main Feature Extraction Logic
def run_feature_extraction(quick_test=False):
    # Check if features already exist to save time
    if os.path.exists('features/X_train.npy') and not quick_test:
        print("Features already exist. Loading from disk...")
        X_train = np.load('features/X_train.npy')
        y_train = np.load('features/y_train.npy')
        X_val = np.load('features/X_validation.npy')
        y_val = np.load('features/y_validation.npy')
        X_test = np.load('features/X_test.npy')
        y_test = np.load('features/y_test.npy')
        return X_train, y_train, X_val, y_val, X_test, y_test

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load pretrained ResNet18
    print("Loading pretrained ResNet18...")
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Identity()  # Remove classification layer
    model = model.to(device)

    batch_size = 64 if not quick_test else 32

    # DataLoaders
    train_loader, val_loader, test_loader = get_data_loaders(batch_size=batch_size)

    os.makedirs('features', exist_ok=True)
    
    features_dict = {}

    # Extract features
    for loader, name in zip([train_loader, val_loader, test_loader], ["Train", "Validation", "Test"]):
        print(f"\nStarting feature extraction for {name} set...")
        X, y = extract_features(loader, model, device, set_name=name, quick=quick_test)
        
        # Save if not quick test (or save anyway if you prefer)
        if not quick_test:
            np.save(f'features/X_{name.lower()}.npy', X)
            np.save(f'features/y_{name.lower()}.npy', y)
        
        features_dict[name] = (X, y)
        print(f"Saved {name} features: X={X.shape}, y={y.shape}")

    print("\nFeature extraction complete.")
    return features_dict['Train'][0], features_dict['Train'][1], features_dict['Validation'][0], features_dict['Validation'][1], features_dict['Test'][0], features_dict['Test'][1]

In [ ]:
# Run extraction (set quick_test=True for debugging, False for full run)
quick_test = False 
X_train, y_train, X_val, y_val, X_test, y_test = run_feature_extraction(quick_test=quick_test)

## 5. Logistic Regression
We train a Logistic Regression classifier on the extracted features. We perform hyperparameter tuning to find the best regularization strength `C`.

In [ ]:
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Hyperparameter Tuning
C_values = [0.01, 0.1, 1, 5, 10]
accuracies = []
best_acc = 0
best_c = C_values[0]

print("Starting Hyperparameter Tuning...")
for c in C_values:
    print(f"Training with C={c}...")
    clf = LogisticRegression(solver='lbfgs', max_iter=2000, C=c, random_state=42)
    clf.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, clf.predict(X_val))
    accuracies.append(val_acc)
    print(f"Validation Accuracy: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_c = c

print(f"Best C: {best_c} with Validation Accuracy: {best_acc:.4f}")

# Plot Accuracy vs C
plot_accuracy_vs_c(C_values, accuracies)

In [ ]:
# Train Final Model on Train + Val
print("Training Final Model on Train + Val...")
X_full_train = np.concatenate((X_train, X_val))
y_full_train = np.concatenate((y_train, y_val))

final_clf = LogisticRegression(solver='lbfgs', max_iter=2000, C=best_c, random_state=42)
final_clf.fit(X_full_train, y_full_train)

# Evaluate on Test
print("Evaluating on Test set...")
y_pred_log = final_clf.predict(X_test)
y_prob_log = final_clf.predict_proba(X_test)

print_metrics(y_test, y_pred_log, classes)

# Plot Confusion Matrix
plot_confusion_matrix(y_test, y_pred_log, classes, title=f'Logistic Regression Confusion Matrix (C={best_c})')

# Plot ROC Curve
plot_roc_curve(y_test, y_prob_log, len(classes), classes, title=f'Logistic Regression ROC Curve (C={best_c})')

## 6. K-Means Clustering
We perform unsupervised clustering using K-Means and map the clusters to the true labels to evaluate performance.

In [ ]:
n_classes = len(classes)

# Train KMeans
print(f"Training KMeans with k={n_classes}...")
kmeans = KMeans(n_clusters=n_classes, n_init=10, random_state=42)
kmeans.fit(X_train)

# Map clusters to labels
print("Mapping clusters to labels...")
train_clusters = kmeans.predict(X_train)
cluster_labels = np.zeros_like(train_clusters)

mapping = {}
for i in range(n_classes):
    mask = (train_clusters == i)
    if np.sum(mask) > 0:
        # Assign the most frequent label in the cluster to the cluster
        most_frequent = mode(y_train[mask], keepdims=True).mode[0]
        mapping[i] = most_frequent
    else:
        mapping[i] = -1

print(f"Cluster Mapping: {mapping}")

# Evaluate on Test
print("Evaluating on Test set...")
test_clusters = kmeans.predict(X_test)
y_pred_kmeans = np.array([mapping[c] for c in test_clusters])

print_metrics(y_test, y_pred_kmeans, classes)

# Silhouette Score
print("Calculating Silhouette Score...")
sil_score = silhouette_score(X_test, test_clusters)
print(f"Silhouette Score: {sil_score:.4f}")

# Plot Confusion Matrix
plot_confusion_matrix(y_test, y_pred_kmeans, classes, title='KMeans Confusion Matrix')